# Results 4 — Variant-level pleiotropy

Cluster counts, the negative-binomial model of the variant pleiotropy score (vPS), the
directionality analysis, and the data behind Figure 3.

| file | panel |
| --- | --- |
| `plot_a.csv` | observed and predicted vPS per MAF bin (Figure 3a) |
| `plot_b.csv` | univariate and joint model coefficients (Figure 3b) |
| `figure_3_apoe.csv` | every disease association of the two APOE variants (Figure 3c) |

Predicted power assumes the variant acts on most traits at an effect size an order of
magnitude below its largest observed effect: the non-centrality parameter is
`maxAbsBeta^2 * maxEffectiveSampleSize * maxVarG / 11`, and power is the survival function of
a non-central chi-square at the genome-wide threshold.

In [1]:
import collections

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import ncx2, spearmanr

from manuscript_methods import clusters, paper

numbers = {}

COVARIATES = [
    "maxAbsBetaNormalised",
    "maxMAFNormalised",
    "maxEffectiveSampleSizeNormalised",
    "gerpNormalisedNormalised",
    "vepBinaryNormalised",
    "predictedPowerNormalised",
]
LABELS = {
    "maxAbsBetaNormalised": "Absolute beta",
    "maxMAFNormalised": "MAF",
    "maxEffectiveSampleSizeNormalised": "Sample size",
    "gerpNormalisedNormalised": "GERP",
    "vepBinaryNormalised": "PAV",
    "predictedPowerNormalised": "Predicted power",
}
MAF_BINS = [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
MAF_BIN_LABELS = ["0-0.01", "0.01-0.05", "0.05-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5"]
GENOME_WIDE_CHI2 = 32.84125  # chi-square value corresponding to p = 5e-8 with 1 degree of freedom

## Clusters

In [2]:
cluster_table = pd.read_parquet(paper.derived("variant_clusters"))

numbers["R4.01"] = len(cluster_table)
numbers["R4.02"] = int((cluster_table["uniqueLeadVariants"] > 1).sum())
numbers["R4.03"] = int((cluster_table["uniqueDiseases"] > 1).sum())
numbers["R4.04"] = int(cluster_table["uniqueDiseases"].max())
numbers["R4.05"] = round(float(cluster_table["uniqueDiseases"].mean()), 2)
numbers["R4.06"] = int((cluster_table["uniqueTherapeuticAreas"] > 1).sum())
numbers["R4.07"] = int(cluster_table["uniqueTherapeuticAreas"].max())
numbers["R4.08"] = round(float(cluster_table["uniqueTherapeuticAreas"].mean()), 2)
rho, pvalue = spearmanr(cluster_table["uniqueDiseases"], cluster_table["uniqueTherapeuticAreas"])
numbers["R4.09"] = round(float(rho), 2)
print({k: numbers[k] for k in sorted(numbers)}, "| Spearman P:", pvalue)

{'R4.01': 20041, 'R4.02': 5595, 'R4.03': 6617, 'R4.04': 120, 'R4.05': 2.14, 'R4.06': 4539, 'R4.07': 20, 'R4.08': 1.4, 'R4.09': 0.81} | Spearman P: 0.0


## Cluster-level covariates

Each cluster is represented by one lead variant: the one associated with most diseases, ties
broken by variant id.

In [3]:
credible_sets = clusters.load_credible_sets()
edges = clusters.load_edges(set(credible_sets["studyLocusId"]))
components = clusters.cluster(list(zip(credible_sets["studyLocusId"], credible_sets["variantId"])), edges)

locus_variant = dict(zip(credible_sets["studyLocusId"], credible_sets["variantId"]))
locus_traits = dict(zip(credible_sets["studyLocusId"], credible_sets["diseaseIds"]))

features = pd.read_parquet(paper.derived("variant_features"))[
    ["variantId", "maxAbsBeta", "maxMAF", "maxEffectiveSampleSize", "maxVarG", "gerpNormalised", "vepScore"]
].drop_duplicates("variantId")

rows = []
for _, members in components:
    per_variant = collections.defaultdict(set)
    for locus_id in members:
        per_variant.setdefault(locus_variant[locus_id], set())
        traits = locus_traits.get(locus_id)
        if traits is not None:
            per_variant[locus_variant[locus_id]].update(traits)
    representative = sorted(per_variant.items(), key=lambda kv: (-len(kv[1]), kv[0]))[0][0]
    all_traits = set().union(*per_variant.values()) if per_variant else set()
    rows.append({"clusterSize": len(members), "vPS": len(all_traits), "clusterVariantId": representative})

frame = pd.DataFrame(rows).merge(features, left_on="clusterVariantId", right_on="variantId", how="inner")
print("clusters joined to variant features:", len(frame), "of", len(components))

clusters joined to variant features: 20041 of 20041


In [4]:
frame["ncp"] = (frame["maxAbsBeta"] ** 2 * frame["maxEffectiveSampleSize"] * frame["maxVarG"]) / 11
frame["predictedPower"] = ncx2.sf(x=GENOME_WIDE_CHI2, df=1, nc=frame["ncp"])
frame["gerpNormalised"] = frame["gerpNormalised"].fillna(frame["gerpNormalised"].mean())
frame["vepBinary"] = (frame["vepScore"] >= 0.66).astype(int)

# Every covariate is min-max scaled so the coefficients are comparable in the forest plot.
for column in ["maxAbsBeta", "maxMAF", "gerpNormalised", "vepBinary", "maxEffectiveSampleSize", "predictedPower"]:
    span = frame[column].max() - frame[column].min()
    frame[f"{column}Normalised"] = 0.0 if span == 0 else (frame[column] - frame[column].min()) / span

## Figure 3b — negative binomial models

In [5]:
def fit(covariates):
    """Negative binomial fit of vPS on the given covariates."""
    x = sm.add_constant(frame[covariates].copy())
    model = sm.NegativeBinomial(frame["vPS"], x).fit(disp=False, maxiter=1000)
    return model, x


records = []
for covariate in COVARIATES:
    model, _ = fit([covariate])
    ci = model.conf_int()
    records.append(
        {
            "covariate": covariate,
            "model_type": "Univariate",
            "coefficient": model.params[covariate],
            "std_error": model.bse[covariate],
            "p_value": model.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

joint, x_joint = fit(COVARIATES)
ci = joint.conf_int()
for covariate in COVARIATES:
    records.append(
        {
            "covariate": covariate,
            "model_type": "Multi",
            "coefficient": joint.params[covariate],
            "std_error": joint.bse[covariate],
            "p_value": joint.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

coefficients = pd.DataFrame(records)
coefficients["covariate_label"] = coefficients["covariate"].map(LABELS)
coefficients["y_numerical"] = coefficients["covariate"].map({c: i for i, c in enumerate(COVARIATES)})
coefficients["y_plot"] = coefficients["y_numerical"] + np.where(coefficients["model_type"] == "Univariate", -0.1, 0.1)
coefficients = coefficients[
    [
        "covariate",
        "covariate_label",
        "model_type",
        "coefficient",
        "std_error",
        "p_value",
        "ci_lower",
        "ci_upper",
        "y_numerical",
        "y_plot",
    ]
]
coefficients.to_csv(paper.derived("plot_b.csv"), index=False)
coefficients.round(4)

,covariate,covariate_label,model_type,coefficient,std_error,p_value,ci_lower,ci_upper,y_numerical,y_plot
0,maxAbsBetaNormalised,Absolute beta,Univariate,1.8767,0.0740,0.0,1.7317,2.0218,0,-0.1
1,maxMAFNormalised,MAF,Univariate,0.3979,0.0230,0.0,0.3529,0.4430,1,0.9
2,maxEffectiveSampleSizeNormalised,Sample size,Univariate,0.8062,0.0397,0.0,0.7285,0.8840,2,1.9
3,gerpNormalisedNormalised,GERP,Univariate,0.4008,0.0273,0.0,0.3474,0.4543,3,2.9
4,vepBinaryNormalised,PAV,Univariate,0.7496,0.0333,0.0,0.6843,0.8149,4,3.9
5,predictedPowerNormalised,Predicted power,Univariate,1.3918,0.0154,0.0,1.3616,1.4220,5,4.9
6,maxAbsBetaNormalised,Absolute beta,Multi,0.3310,0.0751,0.0,0.1838,0.4783,0,0.1
7,maxMAFNormalised,MAF,Multi,0.3559,0.0226,0.0,0.3116,0.4003,1,1.1
8,maxEffectiveSampleSizeNormalised,Sample size,Multi,0.2655,0.0362,0.0,0.1945,0.3364,2,2.1
9,gerpNormalisedNormalised,GERP,Multi,0.1473,0.0266,0.0,0.0953,0.1994,3,3.1


## Variance explained

In [6]:
without_power = [c for c in COVARIATES if c != "predictedPowerNormalised"]
joint_no_power, x_no_power = fit(without_power)
power_only, x_power = fit(["predictedPowerNormalised"])
sample_size_only, x_sample = fit(["maxEffectiveSampleSizeNormalised"])


def r2(model, x):
    """Squared Pearson correlation between observed and predicted vPS."""
    return float(np.corrcoef(frame["vPS"], model.predict(x))[0, 1] ** 2)


numbers["R4.10"] = 100 * r2(power_only, x_power)
numbers["R4.11"] = 100 * r2(joint, x_joint)
numbers["R4.12"] = 100 * r2(joint_no_power, x_no_power)
numbers["R4.13"] = 100 * r2(sample_size_only, x_sample)
print({k: numbers[k] for k in ["R4.10", "R4.11", "R4.12", "R4.13"]})

{'R4.10': 14.668994249704815, 'R4.11': 17.66639579561643, 'R4.12': 5.962180882988754, 'R4.13': 0.4411309867533988}


## Figure 3a — observed and predicted vPS per MAF bin

In [7]:
binned = frame.copy()
binned["predicted_traits_full_model"] = joint.predict(x_joint)
binned["predicted_traits_no_power"] = joint_no_power.predict(x_no_power)
binned["maxMAF_bin"] = pd.cut(binned["maxMAF"], bins=MAF_BINS, labels=MAF_BIN_LABELS, right=False)

bins = (
    binned.groupby("maxMAF_bin", observed=False)
    .agg(
        observed_mean=("vPS", "mean"),
        observed_sem=("vPS", "sem"),
        predicted_full_mean=("predicted_traits_full_model", "mean"),
        predicted_full_sem=("predicted_traits_full_model", "sem"),
        predicted_no_power_mean=("predicted_traits_no_power", "mean"),
        predicted_no_power_sem=("predicted_traits_no_power", "sem"),
    )
    .reset_index()
)
bins["maxMAF_bin"] = bins["maxMAF_bin"].astype(str)
bins.to_csv(paper.derived("plot_a.csv"), index=False)
bins.round(4)

,maxMAF_bin,observed_mean,observed_sem,predicted_full_mean,predicted_full_sem,predicted_no_power_mean,predicted_no_power_sem
0,0-0.01,1.8931,0.1280,2.3544,0.0776,3.1360,0.1082
1,0.01-0.05,1.7753,0.0608,1.8546,0.0266,1.9437,0.0266
2,0.05-0.1,1.9133,0.0711,1.8404,0.0289,1.7582,0.0171
3,0.1-0.2,1.9266,0.0495,1.9019,0.0203,1.7995,0.0102
4,0.2-0.3,2.1608,0.0592,2.0664,0.0214,2.0139,0.0101
5,0.3-0.4,2.2625,0.0549,2.2555,0.0231,2.2767,0.0102
6,0.4-0.5,2.5439,0.0866,2.5079,0.0250,2.5861,0.0101


## Directionality

Concordance is computed on the minor allele, over lead variants associated with more than one
disease and carrying a signed effect.

In [8]:
# lead_vPS is defined on the cluster-representative lead variant, so the directionality
# analysis runs over the representatives, not over every lead variant.
variant_features = pd.read_parquet(paper.derived("variant_features"))
representatives = set(frame["clusterVariantId"])
pleiotropic = variant_features[
    variant_features["variantId"].isin(representatives)
    & (variant_features["uniqueDiseases"] > 1)
    & variant_features["betaSignConcordance"].notna()
]
print("cluster representatives:", len(representatives))

numbers["R4.14"] = len(pleiotropic)
numbers["R4.15"] = int((pleiotropic["betaSignConcordance"] == 1.0).sum())
numbers["R4.16"] = int((pleiotropic["betaSignConcordance"] < 1.0).sum())

highly_pleiotropic = pleiotropic[pleiotropic["uniqueDiseases"] >= 10]
discordant = highly_pleiotropic[highly_pleiotropic["betaSignConcordance"] <= 0.8]
numbers["R4.17"] = len(highly_pleiotropic)
numbers["R4.18"] = len(discordant)
numbers["R4.19"] = len(
    {gene for genes in discordant["prioritisedGenes"] for gene in (genes if genes is not None else [])}
)
print({k: numbers[k] for k in ["R4.14", "R4.15", "R4.16", "R4.17", "R4.18", "R4.19"]})
print("fully concordant share: %.1f%%" % (100 * numbers["R4.15"] / numbers["R4.14"]))

cluster representatives: 20041
{'R4.14': 4568, 'R4.15': 3952, 'R4.16': 616, 'R4.17': 126, 'R4.18': 35, 'R4.19': 38}
fully concordant share: 86.5%


In [9]:
# Supplementary Table 2: the discordant highly pleiotropic lead variants.
st2 = discordant.sort_values("uniqueDiseases", ascending=False)[
    ["variantId", "prioritisedGenes", "uniqueDiseases", "uniqueTherapeuticAreas", "betaSignConcordance", "maxMAF"]
]
st2.to_csv(paper.derived("st2_discordant_variants.csv"), index=False)
st2.head(10)

,variantId,prioritisedGenes,uniqueDiseases,uniqueTherapeuticAreas,betaSignConcordance,maxMAF
11824,19_44908684_T_C,[ENSG00000130203],85,15,0.659459,0.220463
22854,2_27508073_T_C,[ENSG00000084734],34,13,0.627660,0.471640
22467,22_28725099_A_G,[ENSG00000183765],32,4,0.694444,0.025757
17014,1_169549811_C_T,[ENSG00000198734],30,9,0.800000,0.024718
38507,4_99318162_T_C,[ENSG00000196616],29,10,0.680851,0.256431
39671,9_133257521_T_TC,[ENSG00000175164],27,10,0.775000,0.465403
35964,14_94378610_C_T,[ENSG00000197249],26,10,0.731707,0.019394
3725,5_1279675_C_T,[ENSG00000164362],25,5,0.619048,0.409984
35647,12_4275678_T_G,[ENSG00000118971],20,12,0.775000,0.027176
27547,22_43928847_C_G,[ENSG00000100344],20,9,0.714286,0.492750


## The two APOE variants of Figure 3c

In [10]:
APOE_VARIANTS = ["19_44908684_T_C", "19_44908822_C_T"]

apoe = variant_features[variant_features["variantId"].isin(APOE_VARIANTS)].set_index("variantId")
numbers["R4.20"] = int(apoe.loc["19_44908684_T_C", "uniqueDiseases"])
numbers["R4.21"] = round(float(apoe.loc["19_44908684_T_C", "betaSignConcordance"]), 2)
numbers["R4.22"] = int(apoe.loc["19_44908684_T_C", "uniqueTherapeuticAreas"])
print({k: numbers[k] for k in ["R4.20", "R4.21", "R4.22"]})

{'R4.20': 85, 'R4.21': 0.66, 'R4.22': 15}


In [11]:
import pyarrow.compute as pc
import pyarrow.dataset as ds

names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

associations = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(
        columns={
            "variantId": ds.field("variantId"),
            "studyId": ds.field("studyId"),
            "diseaseIds": ds.field("diseaseIds"),
            "originalBeta": ds.field("originalBeta"),
            "estimatedBeta": pc.struct_field(ds.field("rescaledStatistics"), "minorAlleleEstimatedBeta"),
            "pValueMantissa": pc.struct_field(ds.field("variantStatistics"), "pValueMantissa"),
            "pValueExponent": pc.struct_field(ds.field("variantStatistics"), "pValueExponent"),
        },
        filter=pc.field("variantId").isin(APOE_VARIANTS),
    )
    .to_pandas()
)
associations = associations[associations["originalBeta"].notna()].copy()
associations["diseaseNames"] = associations["diseaseIds"].map(
    lambda ids: [names.get(d) for d in (ids if ids is not None else [])]
)
associations["mappedTherapeuticAreas"] = associations["diseaseIds"].map(
    lambda ids: sorted({areas.get(d, "other") for d in (ids if ids is not None else [])})
)
associations["therapeuticAreaNames"] = associations["mappedTherapeuticAreas"].map(
    lambda tas: [paper.THERAPEUTIC_AREAS.get(t, "other") for t in tas]
)
associations["neg_log10_p"] = -(np.log10(associations["pValueMantissa"]) + associations["pValueExponent"])

columns = [
    "studyId",
    "diseaseIds",
    "diseaseNames",
    "mappedTherapeuticAreas",
    "therapeuticAreaNames",
    "estimatedBeta",
    "pValueMantissa",
    "pValueExponent",
    "neg_log10_p",
]
outputs = ["variant_pleiotropy_data_exploded.csv", "variant_pleiotropy_data_exploded_2.csv"]
for variant, name in zip(APOE_VARIANTS, outputs):
    subset = associations[associations["variantId"] == variant].explode("therapeuticAreaNames")
    subset[columns].to_csv(paper.derived(name), index=False)
    print(name, subset.shape)

variant_pleiotropy_data_exploded.csv (192, 11)
variant_pleiotropy_data_exploded_2.csv (59, 11)


## Numbers

In [12]:
print(paper.save_results("variant_pleiotropy", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/variant_pleiotropy.json


,computed
R4.01,20041.000000
R4.02,5595.000000
R4.03,6617.000000
R4.04,120.000000
R4.05,2.140000
R4.06,4539.000000
R4.07,20.000000
R4.08,1.400000
R4.09,0.810000
R4.10,14.668994
